In [1]:
import numpy as np

eps  = 1e-10
kmax = 1000

In [2]:
def find_max_offdiag(A):
    n = A.shape[0]
    p, q = 1, 0
    for i in range(1, n):
        for j in range(0, i):
            if abs(A[i, j]) > abs(A[p, q]):
                p, q = i, j
    return p, q


def compute_cs(A, p, q):
    alpha = (A[p, p] - A[q, q]) / (2.0 * A[p, q])

    if alpha >= 0:
        t = -alpha + np.sqrt(alpha**2 + 1)
    else:
        t = -alpha - np.sqrt(alpha**2 + 1)

    c = 1.0 / np.sqrt(1.0 + t**2)
    s = t * c
    return c, s, t


def jacobi(A, eps=eps, kmax=kmax):
    n    = A.shape[0]
    A    = A.copy().astype(float)
    U    = np.eye(n)
    k    = 0
    p, q = find_max_offdiag(A)

    while abs(A[p, q]) > eps and k <= kmax:
        c, s, t = compute_cs(A, p, q)
        apq = A[p, q]

        for j in range(n):
            if j == p or j == q:
                continue
            apj, aqj = A[p, j], A[q, j]
            A[p, j] = c * apj + s * aqj
            A[j, p] = A[p, j]
            A[q, j] = -s * apj + c * aqj
            A[j, q] = A[q, j]

        A[p, p] = A[p, p] + t * apq
        A[q, q] = A[q, q] - t * apq
        A[p, q] = A[q, p] = 0.0

        for i in range(n):
            uip, uiq = U[i, p], U[i, q]
            U[i, p] =  c * uip + s * uiq
            U[i, q] = -s * uip + c * uiq

        p, q = find_max_offdiag(A)
        k   += 1

    eigenvalues = np.diag(A).copy()
    return eigenvalues, U

In [3]:
matrices = {
    "Ex 1 (lambda = -1, 0, 2)": np.array([[0,0,1],[0,0,1],[1,1,1]], dtype=float),
    "Ex 2 (lambda = -0.828, 0, 4.828)": np.array([[1,1,2],[1,1,2],[2,2,2]], dtype=float),
    "Ex 3 (lambda = 0, 0, 2, 2)": np.array([[1,0,1,0],[0,1,0,1],[1,0,1,0],[0,1,0,1]], dtype=float),
    "Ex 4 (lambda = -1.165, 0, 0, 17.165)": np.array([[1,2,3,4],[2,3,4,5],[3,4,5,6],[4,5,6,7]], dtype=float),
}

for label, A in matrices.items():
    A_init = A.copy()
    eigenvalues, U = jacobi(A)

    idx         = np.argsort(eigenvalues)
    eigenvalues = eigenvalues[idx]
    U           = U[:, idx]
    Lambda      = np.diag(eigenvalues)

    print(f"=== {label} ===")
    print(f"  eigenvalues = {np.round(eigenvalues, 8)}")
    print(f"  ||A_init * U - U * Lambda|| = {np.linalg.norm(A_init @ U - U @ Lambda):.2e}")
    print()

=== Ex 1 (lambda = -1, 0, 2) ===
  eigenvalues = [-1. -0.  2.]
  ||A_init * U - U * Lambda|| = 6.35e-10

=== Ex 2 (lambda = -0.828, 0, 4.828) ===
  eigenvalues = [-0.82842712  0.          4.82842712]
  ||A_init * U - U * Lambda|| = 1.54e-09

=== Ex 3 (lambda = 0, 0, 2, 2) ===
  eigenvalues = [0. 0. 2. 2.]
  ||A_init * U - U * Lambda|| = 0.00e+00

=== Ex 4 (lambda = -1.165, 0, 0, 17.165) ===
  eigenvalues = [-1.16515139 -0.         -0.         17.16515139]
  ||A_init * U - U * Lambda|| = 1.46e-08



In [4]:
def cholesky_iteration(A, eps=eps, kmax=kmax):
    A_prev = A.copy().astype(float)
    A_curr = None
    k      = 0

    while k < kmax:
        try:
            L = np.linalg.cholesky(A_prev)
        except np.linalg.LinAlgError:
            print(f"  Matricea nu este pozitiv definita la iteratia {k}.")
            break

        A_curr = L.T @ L

        if np.linalg.norm(A_curr - A_prev) < eps:
            k += 1
            break

        A_prev = A_curr
        k     += 1

    A_final = A_curr if A_curr is not None else A_prev
    return A_final, k

In [5]:
np.random.seed(42)
n  = 4
M  = np.random.rand(n, n)
Ac = M.T @ M + n * np.eye(n)

Ac2 = np.array([[4, 2, 0],
                [2, 3, 1],
                [0, 1, 2]], dtype=float)

for label, A in [("4x4 random SPD", Ac), ("3x3 pozitiv definita", Ac2)]:
    A_final, k = cholesky_iteration(A)
    print(f"=== {label} ({k} iteratii) ===")
    print("  A_final =\n", np.round(A_final, 8))
    print("  diag(A_final) ≈ valori proprii:", np.round(np.sort(np.diag(A_final))[::-1], 8))
    print()

=== 4x4 random SPD (1000 iteratii) ===
  A_final =
 [[ 8.28701606e+00 -0.00000000e+00 -0.00000000e+00  0.00000000e+00]
 [-0.00000000e+00  4.54967272e+00  1.25800000e-05  0.00000000e+00]
 [-0.00000000e+00  1.25800000e-05  4.44528924e+00 -0.00000000e+00]
 [ 0.00000000e+00  0.00000000e+00 -0.00000000e+00  4.06727908e+00]]
  diag(A_final) ≈ valori proprii: [8.28701606 4.54967272 4.44528924 4.06727908]

=== 3x3 pozitiv definita (57 iteratii) ===
  A_final =
 [[5.66907909 0.         0.        ]
 [0.         2.4760236  0.        ]
 [0.         0.         0.85489731]]
  diag(A_final) ≈ valori proprii: [5.66907909 2.4760236  0.85489731]



In [6]:
def svd_tasks(A):
    p, n = A.shape
    assert p > n

    U, S, VT = np.linalg.svd(A, full_matrices=True)
    V        = VT.T

    singular_values = S
    rank            = int(np.sum(S > eps))
    rank_lib        = np.linalg.matrix_rank(A)
    cond_number     = S[0] / S[S > eps][-1]
    cond_number_lib = np.linalg.cond(A)

    SI = np.zeros((n, p))
    for i in range(rank):
        SI[i, i] = 1.0 / S[i]
    A_I = V @ SI @ U.T

    ATA = A.T @ A
    try:
        A_J       = np.linalg.inv(ATA) @ A.T
        norm_diff = np.linalg.norm(A_I - A_J, ord=1)
    except np.linalg.LinAlgError:
        A_J       = None
        norm_diff = None

    return singular_values, rank, rank_lib, cond_number, cond_number_lib, A_I, A_J, norm_diff

In [7]:
B1 = np.array([[1,2],[3,4],[5,6],[7,8]], dtype=float)

np.random.seed(7)
B2 = np.round(np.random.rand(5, 3) * 10).astype(float)

B3 = np.array([[1,2,3,1],[2,4,1,2],[3,6,2,3],
               [4,8,0,4],[5,10,1,5],[1,2,4,1]], dtype=float)

svd_examples = [
    ("p=4, n=2, rang plin",      B1),
    ("p=5, n=3, aleatoare",      B2),
    ("p=6, n=4, rang deficient", B3),
]

for label, A in svd_examples:
    singular_values, rank, rank_lib, cond_number, cond_number_lib, \
        A_I, A_J, norm_diff = svd_tasks(A)

    print(f"=== {label} ===")
    print(f"  singular_values  = {np.round(singular_values, 6)}")
    print(f"  rank             = {rank}  (numpy: {rank_lib})")
    print(f"  cond_number      = {cond_number:.6f}  (numpy: {cond_number_lib:.6f})")
    print(f"  A_I (Moore-Penrose) =\n{np.round(A_I, 6)}")
    print(f"  ||A_I - pinv(A)|| = {np.linalg.norm(A_I - np.linalg.pinv(A)):.2e}")
    if A_J is not None:
        print(f"  A_J (LS) =\n{np.round(A_J, 6)}")
        print(f"  ||A_I - A_J||_1  = {norm_diff:.2e}")
    else:
        print("  A_J: A^T*A singulara (rang deficient) => formula LS nu se aplica")
    print()

=== p=4, n=2, rang plin ===
  singular_values  = [14.269095  0.626828]
  rank             = 2  (numpy: 2)
  cond_number      = 22.763964  (numpy: 22.763964)
  A_I (Moore-Penrose) =
[[-1.   -0.5   0.    0.5 ]
 [ 0.85  0.45  0.05 -0.35]]
  ||A_I - pinv(A)|| = 1.67e-16
  A_J (LS) =
[[-1.   -0.5   0.    0.5 ]
 [ 0.85  0.45  0.05 -0.35]]
  ||A_I - A_J||_1  = 1.12e-14

=== p=5, n=3, aleatoare ===
  singular_values  = [20.22041   5.955497  3.109191]
  rank             = 3  (numpy: 3)
  cond_number      = 6.503431  (numpy: 6.503431)
  A_I (Moore-Penrose) =
[[-0.093553  0.117221  0.09371  -0.081498  0.052986]
 [ 0.067238  0.09463  -0.04989  -0.054655 -0.051731]
 [ 0.034425 -0.171156  0.00204   0.199261  0.03929 ]]
  ||A_I - pinv(A)|| = 2.19e-17
  A_J (LS) =
[[-0.093553  0.117221  0.09371  -0.081498  0.052986]
 [ 0.067238  0.09463  -0.04989  -0.054655 -0.051731]
 [ 0.034425 -0.171156  0.00204   0.199261  0.03929 ]]
  ||A_I - A_J||_1  = 2.78e-16

=== p=6, n=4, rang deficient ===
  singular_values